# Description

This notebook generates a map of the studies. The output file is in EPS format.

# Requirements

An environment with:
- matplotlib
- pandas
- openpyxl
- geopandas

Use the following to install the packages:

```pip install -e ".[map_articles]"```

# Prerequisites

- An excel file ```filename``` containing the references in ```sheet_name``` with at least the column ```column_name```.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from adjustText import adjust_text


In [ ]:
filename = '../data/refs.xlsx'
sheet_name = 'RAW'
output_filename = "case_studies_map.eps"
column_name = 'Location'
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"


In [ ]:
df = pd.read_excel(filename,sheet_name=sheet_name)
loc = df[column_name].to_list()
countries = pd.DataFrame(loc, columns=["country"])
df = countries.groupby('country').size().reset_index(name='count')

In [ ]:
world = gpd.read_file(url)
world_counts = world.merge(df, left_on="NAME", right_on="country", how="left")
world_counts["count"] = world_counts["count"].fillna(0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
world_counts.plot(
    column="count",       
    cmap="YlGnBu",  
    linewidth=0.8,
    ax=ax,
      edgecolor="grey",
    legend=True,
    legend_kwds={"label": "Number of studies per country",  "shrink": 0.3}
)

cmap = plt.cm.get_cmap("YlGnBu")
norm = Normalize(vmin=world_counts["count"].min(), vmax=world_counts["count"].max())


cbar = ax.get_figure().axes[-1]  
cbar.set_yticks(np.arange(int(world_counts["count"].min()),
                          int(world_counts["count"].max())+1, 1))

texts = []
for idx, row in world_counts.iterrows():
    if row["NAME"] in df["country"].values:
      rgba = cmap(norm(row["count"]))
      luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
      font_color = "white" if luminance < 0.1 else "black"
    
      t = ax.text(row.geometry.centroid.x, row.geometry.centroid.y,
                    row["NAME"], fontsize=10, color=font_color)
      texts.append(t)

adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

plt.savefig(output_filename, format="eps", bbox_inches="tight")

plt.show()